<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 — Modèle de données SQL

Objectif : à partir de la table `inspections` nettoyée, construire un **panel pont × année** avec les événements de dégradation, plus les tables de dimensions.

- **Partie 0** (fournie) : setup + chargement/nettoyage → table `inspections`
- **Étapes 1 à 7** : c'est toi qui écris le SQL

## 0.1 — Setup

In [1]:
!pip install -q jupysql duckdb-engine

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nbi'

import duckdb
con = duckdb.connect(f'{DATA_DIR}/nbi.duckdb')

%load_ext sql
%sql con
%config SqlMagic.displaylimit = 20

Mounted at /content/drive


## 0.2 — Chargement + nettoyage → `inspections`

Deux temps :
1. **Lecture (Python/pandas)** → `inspections_raw` : les 19 colonnes utiles, en texte brut, une année à la fois. On passe par pandas car le lecteur CSV de DuckDB n'arrive pas à lire certains fichiers NBI (fins de ligne mélangées `\n` / `\r\n`, apostrophes dans les textes) ; pandas est plus tolérant.
2. **Nettoyage (SQL)** → `inspections` : typage, conversion lat/lon, notes `'N'` → NULL, `min_rating`, `age`.

| Colonne | Contenu |
|---|---|
| `bridge_id` | `state_code` + `-` + `structure_number` |
| `year` | année du fichier NBI (2016–2025 ; le fichier 2015 téléchargé n'est pas exploitable) |
| `state_code`, `county_code`, `lat`, `lon` | localisation (lat/lon en degrés décimaux, lon négative) |
| `owner_code`, `material_code`, `structure_type_code` | codes bruts (items 22, 43A, 43B) |
| `year_built`, `year_reconstructed` | entiers, NULL si 0/absent |
| `adt`, `pct_truck`, `length_m`, `deck_width_m` | trafic et géométrie |
| `deck`, `super`, `sub`, `culvert` | notes 0–9, `'N'` → NULL |
| `is_culvert` | `culvert IS NOT NULL` |
| `min_rating` | `culvert` si ponceau, sinon `LEAST(deck, super, sub)` |
| `age` | `year - year_built` |

Choix de nettoyage :
- on garde seulement `RECORD_TYPE_005A = '1'` (les lignes de type 2 décrivent la route *sous* le pont et dupliqueraient la structure) ;
- les doublons éventuels ne sont **pas** supprimés ici : tu les chercheras à l'étape 6.

⏱️ Lecture de ~2,5 Go depuis le Drive : compte plusieurs minutes.

In [2]:
# Lecture des fichiers bruts avec pandas, un fichier (= une année) à la fois.
# pandas tolère les fins de ligne mélangées (\n / \r\n) qui font planter le lecteur CSV de DuckDB.
import glob, os, re
import pandas as pd

COLS = [
    'STATE_CODE_001', 'STRUCTURE_NUMBER_008', 'RECORD_TYPE_005A', 'COUNTY_CODE_003',
    'LAT_016', 'LONG_017', 'OWNER_022', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B',
    'YEAR_BUILT_027', 'YEAR_RECONSTRUCTED_106', 'ADT_029', 'PERCENT_ADT_TRUCK_109',
    'STRUCTURE_LEN_MT_049', 'DECK_WIDTH_MT_052',
    'DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062',
]

con.execute('DROP TABLE IF EXISTS inspections_raw')
for f in sorted(glob.glob(f'{DATA_DIR}/raw/nbi_20*.txt')):
    year = int(re.search(r'nbi_(20\d{2})', f).group(1))
    if year < 2016:          # nbi_2015.txt n'est pas un vrai fichier NBI
        continue
    df = pd.read_csv(
        f, sep=',', quotechar="'", encoding='latin-1',
        dtype=object, keep_default_na=False, na_values=[''],   # tout en texte, vide -> NULL
        usecols=lambda c: c.strip() in COLS,
        on_bad_lines='warn',                                # signale les lignes illisibles au lieu de planter
    )
    df.columns = df.columns.str.strip()
    df = df.reindex(columns=COLS)                           # colonne absente une année -> NULL
    df.insert(0, 'year', year)
    if con.sql("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = 'inspections_raw'").fetchone()[0] == 0:
        con.execute('CREATE TABLE inspections_raw AS SELECT * FROM df')
    else:
        con.execute('INSERT INTO inspections_raw SELECT * FROM df')
    print(f'{year} : {len(df):>8,} lignes')
    del df

print(con.sql('SELECT COUNT(*) AS total FROM inspections_raw').fetchone()[0], 'lignes au total')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2016 :  614,387 lignes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2017 :  615,002 lignes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

2023 :  621,581 lignes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2024 :  623,218 lignes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025 :  624,193 lignes
6190306 lignes au total


In [3]:
%%sql
-- Nettoyage + typage : inspections_raw (texte brut) -> inspections (table propre)
CREATE OR REPLACE TABLE inspections AS
WITH typed AS (
    SELECT
        lpad(trim(STATE_CODE_001), 2, '0') || '-' || trim(STRUCTURE_NUMBER_008) AS bridge_id,
        CAST(year AS INTEGER) AS year,
        lpad(trim(STATE_CODE_001), 2, '0')  AS state_code,
        lpad(trim(COUNTY_CODE_003), 3, '0') AS county_code,

        -- LAT_016 = DDMMSSss, LONG_017 = DDDMMSSss (ouest, donc négatif)
        lpad(trim(LAT_016), 8, '0')  AS lat_txt,
        lpad(trim(LONG_017), 9, '0') AS lon_txt,

        lpad(trim(OWNER_022), 2, '0')            AS owner_code,
        trim(STRUCTURE_KIND_043A)                AS material_code,
        lpad(trim(STRUCTURE_TYPE_043B), 2, '0')  AS structure_type_code,

        NULLIF(TRY_CAST(trim(YEAR_BUILT_027)         AS INTEGER), 0) AS year_built,
        NULLIF(TRY_CAST(trim(YEAR_RECONSTRUCTED_106) AS INTEGER), 0) AS year_reconstructed,

        TRY_CAST(trim(ADT_029)               AS INTEGER) AS adt,
        TRY_CAST(trim(PERCENT_ADT_TRUCK_109) AS INTEGER) AS pct_truck,
        TRY_CAST(trim(STRUCTURE_LEN_MT_049)  AS DOUBLE)  AS length_m,
        TRY_CAST(trim(DECK_WIDTH_MT_052)     AS DOUBLE)  AS deck_width_m,

        -- notes 0-9 ; 'N' (non applicable) ou vide -> NULL
        TRY_CAST(trim(DECK_COND_058)           AS TINYINT) AS deck,
        TRY_CAST(trim(SUPERSTRUCTURE_COND_059) AS TINYINT) AS super,
        TRY_CAST(trim(SUBSTRUCTURE_COND_060)   AS TINYINT) AS sub,
        TRY_CAST(trim(CULVERT_COND_062)        AS TINYINT) AS culvert
    FROM inspections_raw
    WHERE trim(RECORD_TYPE_005A) = '1'   -- 1 = la structure elle-même (2 = route qui passe dessous, doublon)
)
SELECT
    bridge_id, year, state_code, county_code,
    CASE WHEN TRY_CAST(lat_txt AS BIGINT) > 0 THEN
        CAST(substr(lat_txt, 1, 2) AS INTEGER)
      + CAST(substr(lat_txt, 3, 2) AS INTEGER) / 60.0
      + CAST(substr(lat_txt, 5, 4) AS INTEGER) / 100.0 / 3600.0
    END AS lat,
    CASE WHEN TRY_CAST(lon_txt AS BIGINT) > 0 THEN
      -( CAST(substr(lon_txt, 1, 3) AS INTEGER)
       + CAST(substr(lon_txt, 4, 2) AS INTEGER) / 60.0
       + CAST(substr(lon_txt, 6, 4) AS INTEGER) / 100.0 / 3600.0 )
    END AS lon,
    owner_code, material_code, structure_type_code,
    year_built, year_reconstructed,
    adt, pct_truck, length_m, deck_width_m,
    deck, super, sub, culvert,
    culvert IS NOT NULL AS is_culvert,
    CASE WHEN culvert IS NOT NULL THEN culvert
         ELSE LEAST(deck, super, sub)          -- LEAST ignore les NULL dans DuckDB
    END AS min_rating,
    year - year_built AS age
FROM typed;

Running query in 'DuckDBPyConnection'

Count
6190306


In [4]:
%%sql
-- Vérification rapide du chargement (pas l'étape 1 : juste s'assurer que la table existe)
SELECT COUNT(*) AS n_rows, MIN(year) AS first_year, MAX(year) AS last_year
FROM inspections;

Running query in 'DuckDBPyConnection'

n_rows,first_year,last_year
6190306,2016,2025


In [5]:
%%sql
SELECT * FROM inspections LIMIT 5;

Running query in 'DuckDBPyConnection'

bridge_id,year,state_code,county_code,lat,lon,owner_code,material_code,structure_type_code,year_built,year_reconstructed,adt,pct_truck,length_m,deck_width_m,deck,super,sub,culvert,is_culvert,min_rating,age
01-00000000000S702,2016,01,053,31.103083333333334,-87.57038888888889,62,5,05,1999,None,50,1,15.0,9.7,8,8,7,None,False,7,17
01-00000000000S703,2016,01,053,31.105611111111113,-87.56913888888889,62,5,01,2002,None,159,1,13.6,9.8,8,8,7,None,False,7,14
01-0000000000M0022,2016,01,113,32.29535555555555,-84.97722222222222,74,1,04,1942,None,375,35,43.0,9.0,5,5,6,None,False,5,74
01-000000883039900,2016,01,059,34.45166666666667,-87.97,67,5,05,1974,None,300,5,65.5,10.5,7,7,7,None,False,7,42
01-000001014002450,2016,01,079,34.805,-87.38166666666666,67,3,10,1937,1962,6200,10,1981.2,7.8,6,6,7,None,False,6,79


In [6]:
%%sql
-- SELECT * FROM (describe inspections) ;
summarize inspections

Running query in 'DuckDBPyConnection'

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
bridge_id,VARCHAR,01-00000000000S702,78-STX8205,618519,None,None,None,None,None,6190306,0.00
year,INTEGER,2016,2025,11,2020.5149522495333,2.872505728226467,2018,2021,2023,6190306,0.00
state_code,VARCHAR,01,78,57,None,None,None,None,None,6190306,0.00
county_code,VARCHAR,001,840,330,None,None,None,None,None,6190306,0.00
lat,DOUBLE,8.333333333333334e-06,100.67777500000001,963509,37.90582099365097,5.072872148217413,34.42460031850702,38.49205287331214,41.321834822630265,6190306,0.06
lon,DOUBLE,-1000.677775,-5.555555555555556e-06,994436,-91.80387881350525,14.116938991824476,-97.14747449001985,-90.2403847491525,-83.02743621251423,6190306,0.13
owner_code,VARCHAR,00,80,41,None,None,None,None,None,6190306,0.00
material_code,VARCHAR,0,A,11,None,None,None,None,None,6190306,0.00
structure_type_code,VARCHAR,00,22,23,None,None,None,None,None,6190306,0.00
year_built,INTEGER,2,2026,283,1975.3424833285947,26.352013807863013,1959,1976,1996,6190306,0.00


---
## Étape 1 — Prise en main de la table (15 min)

**Objectif :** répondre à 4 questions, une requête chacune.
1. Combien de lignes (= ponts inspectés) par année ?
2. Combien de ponts **distincts** au total sur toute la période ?
3. Distribution de `min_rating` par année (combien de ponts à 0, 1, …, 9 chaque année).
4. Part de ponceaux (`is_culvert`) — globale ou par année.

**Concepts :** `GROUP BY`, `COUNT(DISTINCT …)`, agrégats conditionnels `COUNT(*) FILTER (WHERE …)`.

**Indices (sans solution) :**
- Q1 et Q2 ne donnent pas le même ordre de grandeur que la somme des lignes : pourquoi ?
- Q3 : deux façons possibles — format « long » (`GROUP BY` sur deux colonnes) ou format « large » (une colonne par note avec `FILTER`). Essaie au moins le format large pour pratiquer `FILTER`.
- Q4 : un booléen se compte avec `FILTER (WHERE is_culvert)`, et une part = compte filtré / compte total. Attention à la division entière.
- Pense aux `NULL` dans `min_rating` : où vont-ils dans tes résultats ?

In [7]:
%%sql
-- Number of bridges inspected by year
SELECT COUNT (DISTINCT bridge_id) AS nb_bridges_pery, year FROM inspections
GROUP BY year,
ORDER BY year;

Running query in 'DuckDBPyConnection'

nb_bridges_pery,year
614383,2016
614998,2017
616092,2018
617078,2019
618455,2020
619620,2021
620666,2022
621579,2023
623216,2024
624191,2025


In [8]:
%%sql
-- Nb of bridges
SELECT COUNT (DISTINCT bridge_id) AS distinct_bridges FROM inspections;


Running query in 'DuckDBPyConnection'

distinct_bridges
662768


In [9]:
%%sql
-- Nb of bridges with a min rate per year
SELECT year,
COUNT (min_rating) FILTER  (WHERE min_rating = 0) AS minrat_0,
COUNT (min_rating) FILTER  (WHERE min_rating = 1) AS minrat_1,
COUNT (min_rating) FILTER  (WHERE min_rating = 2) AS minrat_2,
COUNT (min_rating) FILTER  (WHERE min_rating = 3) AS minrat_3,
COUNT (min_rating) FILTER  (WHERE min_rating = 4) AS minrat_4,
COUNT (min_rating) FILTER  (WHERE min_rating = 5) AS minrat_5,
COUNT (min_rating) FILTER  (WHERE min_rating = 6) AS minrat_6,
COUNT (min_rating) FILTER  (WHERE min_rating = 7) AS minrat_7,
COUNT (min_rating) FILTER  (WHERE min_rating = 8) AS minrat_8,
COUNT (min_rating) FILTER  (WHERE min_rating = 9) AS minrat_9
FROM inspections
GROUP BY year,
ORDER BY year;



Running query in 'DuckDBPyConnection'

year,minrat_0,minrat_1,minrat_2,minrat_3,minrat_4,minrat_5,minrat_6,minrat_7,minrat_8,minrat_9
2016,1064,472,2052,9412,35553,102861,171441,209330,70256,11835
2017,1071,479,1901,8984,35181,105218,174047,210340,66523,11174
2018,980,444,1804,8802,35021,109250,176421,210220,62373,10730
2019,971,413,1736,8394,34647,111567,179765,209777,59384,10428
2020,967,440,1671,8124,33827,112796,182191,211548,56813,10079
2021,898,426,1560,7895,32805,113387,184516,214233,54412,9490
2022,1147,393,1310,7627,32490,114094,187294,214899,52542,8873
2023,769,396,1248,7430,32560,114488,189568,215621,50990,8511
2024,799,412,1250,7341,32276,114084,192190,216500,49643,8723
2025,764,431,1209,7118,32160,112275,197447,215168,48731,8890


In [10]:
%%sql
-- Q4 — part de ponceaux
SELECT COUNT (DISTINCT bridge_id) AS per_culvert_pery, year FROM inspections
GROUP BY year,
ORDER BY year;


Running query in 'DuckDBPyConnection'

per_culvert_pery,year
614383,2016
614998,2017
616092,2018
617078,2019
618455,2020
619620,2021
620666,2022
621579,2023
623216,2024
624191,2025


In [12]:
%%sql
-- Percentage of culvert per year
SELECT ROUND( COUNT (is_culvert) FILTER (WHERE is_culvert==1) / COUNT (*) *100,2) AS per_culvert_pery, year FROM inspections
GROUP BY year,
ORDER BY year;


Running query in 'DuckDBPyConnection'

per_culvert_pery,year
22.51,2016
22.68,2017
22.83,2018
22.97,2019
23.13,2020
23.31,2021
23.45,2022
23.58,2023
23.7,2024
23.8,2025


Je regarde d'abord où en est le notebook et quels codes existent dans les données, pour que l'indice colle à ta table.

# Étape 2 — Tables de lookup

Je n'ai pas modifié le notebook, puisque tu m'as demandé de ne pas toucher à `notebooks/`. Tout est ci-dessous, prêt à coller dans `02_sql_model.ipynb`.

## Retour rapide sur l'étape 1

- **Q2, Q3 et le 2ᵉ Q4 sont bons.** En DuckDB, `/` fait une vraie division (pas de troncature), donc ta part de ponceaux est juste. Tu peux écrire simplement `FILTER (WHERE is_culvert)`, car c'est déjà un booléen.
- **Q1 cache quelque chose.** Tu as compté `COUNT(DISTINCT bridge_id)` et tu obtiens 614 383 pour 2016, alors que le chargement indiquait 614 387 lignes. Il y a donc environ 4 doublons `bridge_id × year` par an. La question demandait le nombre de lignes, donc `COUNT(*)`. Note cet écart, tu le retrouveras à l'étape 6.
- **Ta première cellule Q4 est une copie de Q1.** Tu peux la supprimer.
- **Q3 :** ton tableau n'affiche pas les `min_rating` NULL. Combien y en a-t-il ? Une colonne `COUNT(*) FILTER (WHERE min_rating IS NULL)` te le dira.

---

## Objectif

Créer trois petites tables de dimensions qui traduisent les codes bruts en libellés lisibles, puis vérifier que **chaque code présent dans `inspections` existe bien dans sa dimension**.

| Table | Clé (doit correspondre à) | Colonnes suggérées |
|---|---|---|
| `dim_material` | `inspections.material_code` (1 caractère) | `material_code`, `material_label`, `material_family` |
| `dim_owner` | `inspections.owner_code` (2 caractères) | `owner_code`, `owner_label`, `owner_group` |
| `dim_state` | `inspections.state_code` (2 caractères) | `state_code`, `state_abbr`, `state_name` |

Les colonnes `*_family` et `*_group` regroupent les codes en catégories plus larges. Elles serviront dans Power BI, où 30 types de propriétaires seraient illisibles.

## Données de référence (Coding Guide FHWA)

Ce sont des données, pas la solution : c'est toi qui écris le SQL autour.

**Item 43A — matériau**
```
1 Concrete | 2 Concrete continuous | 3 Steel | 4 Steel continuous
5 Prestressed concrete | 6 Prestressed concrete continuous
7 Wood or Timber | 8 Masonry | 9 Aluminum, Wrought or Cast Iron | 0 Other
```

**Item 22 — propriétaire**
```
01 State Highway Agency | 02 County Highway Agency | 03 Town or Township Highway Agency
04 City or Municipal Highway Agency | 11 State Park, Forest, or Reservation Agency
12 Local Park, Forest, or Reservation Agency | 21 Other State Agencies | 25 Other Local Agencies
26 Private (other than railroad) | 27 Railroad | 31 State Toll Authority | 32 Local Toll Authority
60 Other Federal Agencies | 61 Indian Tribal Government | 62 Bureau of Indian Affairs
63 Bureau of Fish and Wildlife | 64 U.S. Forest Service | 66 National Park Service
67 Tennessee Valley Authority | 68 Bureau of Land Management | 69 Bureau of Reclamation
70 Corps of Engineers (Civil) | 71 Corps of Engineers (Military) | 72 Air Force
73 Navy/Marines | 74 Army | 75 NASA | 76 Metropolitan Washington Airports Service | 80 Unknown
```

**Item 1 — état (code FIPS)**
```
01 AL Alabama | 02 AK Alaska | 04 AZ Arizona | 05 AR Arkansas | 06 CA California | 08 CO Colorado
09 CT Connecticut | 10 DE Delaware | 11 DC District of Columbia | 12 FL Florida | 13 GA Georgia
15 HI Hawaii | 16 ID Idaho | 17 IL Illinois | 18 IN Indiana | 19 IA Iowa | 20 KS Kansas
21 KY Kentucky | 22 LA Louisiana | 23 ME Maine | 24 MD Maryland | 25 MA Massachusetts
26 MI Michigan | 27 MN Minnesota | 28 MS Mississippi | 29 MO Missouri | 30 MT Montana
31 NE Nebraska | 32 NV Nevada | 33 NH New Hampshire | 34 NJ New Jersey | 35 NM New Mexico
36 NY New York | 37 NC North Carolina | 38 ND North Dakota | 39 OH Ohio | 40 OK Oklahoma
41 OR Oregon | 42 PA Pennsylvania | 44 RI Rhode Island | 45 SC South Carolina | 46 SD South Dakota
47 TN Tennessee | 48 TX Texas | 49 UT Utah | 50 VT Vermont | 51 VA Virginia | 53 WA Washington
54 WV West Virginia | 55 WI Wisconsin | 56 WY Wyoming | 72 PR Puerto Rico
```
Cette liste est volontairement incomplète. Ton contrôle d'orphelins doit te dire ce qui manque (indice : ton `LIMIT 5` montrait un `bridge_id` qui commence par `78-`).

## Concepts SQL

- `CREATE OR REPLACE TABLE … AS SELECT … FROM (VALUES …)`
- `LEFT JOIN` + `WHERE … IS NULL` (anti-jointure), ou `NOT EXISTS`
- `DISTINCT` / `GROUP BY` pour lister les codes réellement présents

## Indices (sans solution)

1. **Syntaxe.** `VALUES` s'utilise comme une table dans un `FROM`. Ses colonnes n'ont pas de nom, alors tu les nommes avec un alias de la forme `AS v(col_a, col_b, col_c)`. Utilise `CREATE OR REPLACE` pour pouvoir relancer la cellule.
2. **Piège du type de clé.** `inspections.owner_code` est un texte complété à 2 caractères (`'01'`, pas `1`). Si tu écris `(1, 'State Highway Agency')`, la jointure ne trouvera rien. Ta clé doit avoir exactement le même type et le même format que la colonne de `inspections`. Même chose pour `material_code` : 1 caractère, en texte.
3. **Anti-jointure.** Pars de `inspections` (à gauche), joins la dimension, puis garde les lignes où la clé côté dimension est NULL. Pour que ce soit utile, renvoie **quels codes** sont orphelins et **combien de lignes** chacun touche, pas seulement un total.
4. **NULL ou orphelin ?** Une ligne où `owner_code` est lui-même NULL ressort aussi de ton anti-jointure, mais ce n'est pas un code manquant dans la dimension. Sépare les deux cas.
5. **Bonus :** fais le contrôle dans l'autre sens. Quels codes de la dimension n'apparaissent jamais dans les données ? Ce n'est pas une erreur, mais c'est bon à savoir.

## Ce que tu dois produire

- 3 cellules `CREATE TABLE`, une par dimension
- 3 requêtes de contrôle d'orphelins, ou une seule avec `UNION ALL` si tu veux pratiquer
- Si des codes sont orphelins, corrige la dimension et relance le contrôle jusqu'à obtenir 0 code non mappé (hors NULL)

Colle tes requêtes ici quand tu es prêt·e. Je les relis et je te dis ce qui cloche. Les données sont sur ton Drive, donc je ne peux pas les exécuter moi-même : envoie-moi aussi la sortie de tes contrôles d'orphelins.

In [43]:
%%sql
CREATE OR REPLACE TABLE dim_material(
    material_code	VARCHAR,
    material_label VARCHAR
);

INSERT INTO dim_material
  (material_code, material_label)
VALUES
    ('1', 'Concrete'),
    ('2', 'Concrete continuous'),
    ('3', 'Steel'),
    ('4', 'Steel continuous'),
    ('5', 'Prestressed concrete'),
    ('6', 'Prestressed concrete continuous'),
    ('7', 'Wood or Timber'),
    ('8', 'Masonry'),
    ('9', 'Aluminum, Wrought Iron or Cast Iron'),
    ('0', 'Other');



Running query in 'DuckDBPyConnection'

Count
10


In [44]:
%%sql
select * from dim_material;

Running query in 'DuckDBPyConnection'

material_code,material_label
1,Concrete
2,Concrete continuous
3,Steel
4,Steel continuous
5,Prestressed concrete
6,Prestressed concrete continuous
7,Wood or Timber
8,Masonry
9,"Aluminum, Wrought Iron or Cast Iron"
0,Other


In [49]:
%%sql
CREATE OR REPLACE TABLE dim_owner(
    owner_code	VARCHAR,
    owner_label VARCHAR
);

INSERT INTO dim_owner
  (owner_code, owner_label)
VALUES
    ('01', 'State Highway Agency'),
    ('02', 'County Highway Agency'),
    ('03', 'Town or Township Highway Agency'),
    ('04', 'City or Municipal Highway Agency'),
    ('11', 'State Park, Forest, or Reservation Agency'),
    ('12', 'Local Park, Forest, or Reservation Agency'),
    ('21', 'Other State Agencies'),
    ('25', 'Other Local Agencies'),
    ('26', 'Private (other than railroad)'),
    ('27', 'Railroad'),
    ('31', 'State Toll Authority'),
    ('32', 'Local Toll Authority'),
    ('60', 'Other Federal Agencies'),
    ('61', 'Indian Tribal Government'),
    ('62', 'Bureau of Indian Affairs'),
    ('63', 'Bureau of Fish and Wildlife'),
    ('64', 'U.S. Forest Service'),
    ('66', 'National Park Service'),
    ('67', 'Tennessee Valley Authority'),
    ('68', 'Bureau of Land Management'),
    ('69', 'Bureau of Reclamation'),
    ('70', 'Corps of Engineers (Civil)'),
    ('71', 'Corps of Engineers (Military)'),
    ('72', 'Air Force'),
    ('73', 'Navy/Marines'),
    ('74', 'Army'),
    ('75', 'NASA'),
    ('76', 'Metropolitan Washington Airports Service'),
    ('80', 'Unknown');

Running query in 'DuckDBPyConnection'

Count
29


In [50]:
%%sql
select * from dim_owner;

Running query in 'DuckDBPyConnection'

owner_code,owner_label
01,State Highway Agency
02,County Highway Agency
03,Town or Township Highway Agency
04,City or Municipal Highway Agency
11,"State Park, Forest, or Reservation Agency"
12,"Local Park, Forest, or Reservation Agency"
21,Other State Agencies
25,Other Local Agencies
26,Private (other than railroad)
27,Railroad


In [51]:
%%sql
CREATE OR REPLACE TABLE dim_state (
    state_code VARCHAR,
    state_abbr VARCHAR,
    state_name VARCHAR
);

INSERT INTO dim_state (state_code, state_abbr, state_name)
VALUES
    ('01', 'AL', 'Alabama'),
    ('02', 'AK', 'Alaska'),
    ('04', 'AZ', 'Arizona'),
    ('05', 'AR', 'Arkansas'),
    ('06', 'CA', 'California'),
    ('08', 'CO', 'Colorado'),
    ('09', 'CT', 'Connecticut'),
    ('10', 'DE', 'Delaware'),
    ('11', 'DC', 'District of Columbia'),
    ('12', 'FL', 'Florida'),
    ('13', 'GA', 'Georgia'),
    ('15', 'HI', 'Hawaii'),
    ('16', 'ID', 'Idaho'),
    ('17', 'IL', 'Illinois'),
    ('18', 'IN', 'Indiana'),
    ('19', 'IA', 'Iowa'),
    ('20', 'KS', 'Kansas'),
    ('21', 'KY', 'Kentucky'),
    ('22', 'LA', 'Louisiana'),
    ('23', 'ME', 'Maine'),
    ('24', 'MD', 'Maryland'),
    ('25', 'MA', 'Massachusetts'),
    ('26', 'MI', 'Michigan'),
    ('27', 'MN', 'Minnesota'),
    ('28', 'MS', 'Mississippi'),
    ('29', 'MO', 'Missouri'),
    ('30', 'MT', 'Montana'),
    ('31', 'NE', 'Nebraska'),
    ('32', 'NV', 'Nevada'),
    ('33', 'NH', 'New Hampshire'),
    ('34', 'NJ', 'New Jersey'),
    ('35', 'NM', 'New Mexico'),
    ('36', 'NY', 'New York'),
    ('37', 'NC', 'North Carolina'),
    ('38', 'ND', 'North Dakota'),
    ('39', 'OH', 'Ohio'),
    ('40', 'OK', 'Oklahoma'),
    ('41', 'OR', 'Oregon'),
    ('42', 'PA', 'Pennsylvania'),
    ('44', 'RI', 'Rhode Island'),
    ('45', 'SC', 'South Carolina'),
    ('46', 'SD', 'South Dakota'),
    ('47', 'TN', 'Tennessee'),
    ('48', 'TX', 'Texas'),
    ('49', 'UT', 'Utah'),
    ('50', 'VT', 'Vermont'),
    ('51', 'VA', 'Virginia'),
    ('53', 'WA', 'Washington'),
    ('54', 'WV', 'West Virginia'),
    ('55', 'WI', 'Wisconsin'),
    ('56', 'WY', 'Wyoming'),
    ('66', 'GU', 'Guam'),
    ('72', 'PR', 'Puerto Rico'),
    ('78', 'VI', 'U.S. Virgin Islands');

Running query in 'DuckDBPyConnection'

Count
54
